In [1]:
import deeplabcut
import glob
import os
import numpy as np
import pandas as pd

# add path to DLC_preprocessing.py
import sys
sys.path.append(r"C:\Users\schafferlab\github\SNLab_ephys\preprocessing\behavior")
from DLC_preprocessing import update_config_cropping, strip_dlc_suffix
from dlc_crop_selector import get_video_bounds, select_crop_and_update_csv

Loading DLC 2.3.9...


### Create check_dlc.csv from csv file 

csv file contains single column titled 'basepath' of candidate basepaths to include in analysis

In [ ]:
basepaths = pd.read_csv(r"Y:\laura_berkowitz\behavior_validation\appps1_cpp\session_csv\candidate_sessions.csv") 

# get all csv files in basepath that have DLC in the name 
check_dlc = pd.DataFrame(columns=['basepath','video_file'])
for basepath in basepaths['basepath'].values:
    video_files = glob.glob(os.path.join(basepath, "*.avi"))

    temp_df = pd.DataFrame(video_files, columns=['video_file'])
    temp_df['basepath'] = basepath
    check_dlc = pd.concat([check_dlc, temp_df], ignore_index=True)
    check_dlc["redo"] = True
    check_dlc["config_path"] = 'enter config path here'  # placeholder, will be updated later
    check_dlc["y_min"] = 0
    check_dlc["y_max"] = 0
    check_dlc["x_min"] = 0
    check_dlc["x_max"] = 0

    # update columns for cropping bounds with the max possible values (i.e. no crop) this will be overwritten later
    for video in video_files:
        y_min, y_max, x_min, x_max = get_video_bounds(video)
        check_dlc.loc[check_dlc["video_file"] == video, "y_min"] = y_min
        check_dlc.loc[check_dlc["video_file"] == video, "y_max"] = y_max
        check_dlc.loc[check_dlc["video_file"] == video, "x_min"] = x_min
        check_dlc.loc[check_dlc["video_file"] == video, "x_max"] = x_max


### Create check_dlc.csv from directory 

In [2]:
data_dirs = [
    r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_2",
    r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_3",
    r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_4",
]

# use glob to get all video files in the data directories
videos = [fo for data_dir in data_dirs for fo in glob.glob(os.path.join(data_dir, "**\*.avi"), recursive=True)]

# create a dataframe to keep track of which videos need to be redone and their cropping bounds
check_dlc = pd.DataFrame({"basepath": [os.path.dirname(fo) for fo in videos], "video_file": videos})
check_dlc["redo"] = True
check_dlc["config_path"] = 'enter config path here'  # placeholder, will be updated later
check_dlc["y_min"] = 0
check_dlc["y_max"] = 0
check_dlc["x_min"] = 0
check_dlc["x_max"] = 0

# update columns for cropping bounds with the max possible values (i.e. no crop) this will be overwritten later
for video in videos:
    y_min, y_max, x_min, x_max = get_video_bounds(video)
    check_dlc.loc[check_dlc["video_file"] == video, "y_min"] = y_min
    check_dlc.loc[check_dlc["video_file"] == video, "y_max"] = y_max
    check_dlc.loc[check_dlc["video_file"] == video, "x_min"] = x_min
    check_dlc.loc[check_dlc["video_file"] == video, "x_max"] = x_max

### Save check_dlc.csv to preprocessing file in data directory


In [3]:
check_dlc.to_csv(r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\preprocessing\check_dlc.csv", index=False)

### Analyze videos using a csv 'check_dlc.csv' 

check_dlc.csv contains:

```
basepath : str (path to video file)
dlc_file: str (path to video file to analyze or DLC file (current))
redo: bool (indicates if you want to analyze the video)
config_path: str (path to the DLC config file for the appropriate DLC model)
y_min: int|numeric (indicates y_min value for cropping, in pixels)
y_max: int|numeric (indicates y_max value for cropping, in pixels)
x_min: int|numeric (indicates x_min value for cropping, in pixels)
x_max: int|numeric (indicates x_max value for cropping, in pixels)

### Interactive crop selection and update of check_dlc.csv

Use this section to select crop bounds directly in the notebook from a video frame.

Workflow:
1. provide `check_csv_path` and either `video_file` or `basepath`
2. preview `middle`, `mean`, or `both` frame modes
3. click four points around the valid arena region
4. write `x_min`, `x_max`, `y_min`, `y_max` back to the matching row in `check_dlc.csv`

In [ ]:
# import sys
# sys.path.append(r"C:\Users\schafferlab\github\SNLab_ephys\preprocessing\behavior")
# from DLC_preprocessing import update_config_cropping, strip_dlc_suffix
# from dlc_crop_selector import *

# Example: interactively update one row by video_file
check_csv_path = r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\preprocessing\check_dlc.csv"

check_dlc = pd.read_csv(check_csv_path)
check_dlc = check_dlc.query("redo == 1")

for video_file in check_dlc["video_file"].values:
    print(f"Processing {video_file}...")
    result = select_crop_and_update_csv(
        check_csv_path=check_csv_path,
        video_file=video_file,
        preview_mode="mean",  # 'middle', 'mean', or 'both'
        n_samples=100,
        save=True,
        backup=True,
        show_overlay=True,
        update_dlc_config=False,
        overwrite=True,
    )



### Analyze videos in check_dlc.csv

In [ ]:
basepaths = pd.read_csv(r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\preprocessing\check_dlc.csv")
# restrict to rows where redo == 1 and y_min is not null
basepaths = basepaths.query("redo == 1")
basepaths

In [2]:
# load check_dlc.csv 
basepaths = pd.read_csv(r"Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\preprocessing\check_dlc.csv")
# restrict to rows where redo == 1 and y_min is not null
basepaths = basepaths.query("redo == 1")

# loop in unique config_path and video_file
for i, video_file in zip(basepaths.index, basepaths["video_file"].values):
        
    config_path = os.path.normpath(basepaths.loc[i, "config_path"])
    xmin = int(basepaths.loc[i, "x_min"])
    xmax = int(basepaths.loc[i, "x_max"])
    ymin = int(basepaths.loc[i, "y_min"])
    ymax = int(basepaths.loc[i, "y_max"])

    update_config_cropping(config_path, xmin, xmax, ymin, ymax)

    deeplabcut.analyze_videos(
        config_path,
        [video_file],
        videotype="avi",
        shuffle=1,
        trainingsetindex=0,
        save_as_csv=True,
        dynamic=(False, 0.6, 100),
    )

    # deeplabcut.filterpredictions(config_path, video_files)
    deeplabcut.filterpredictions(config_path, [video_file])

    print (f"Finished processing {video_file}")

    basepaths.loc[i, "redo"] = False

Cropping parameters already set to (0, 1440, 0, 1080). No update needed.
Using snapshot-115000 for model Y:\laura_berkowitz\dlc_models\cheeseboard_rig2-andy-2025-07-30\dlc-models\iteration-0\cheeseboard_rig2Jul30-trainset95shuffle1


c:\Users\schafferlab\anaconda3\envs\DEEPLABCUT\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Starting to analyze %  Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_2\4441\4441_exposure2_phase3\4441_newports2_day1_learn-08212025094709-0000.avi
Loading  Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_2\4441\4441_exposure2_phase3\4441_newports2_day1_learn-08212025094709-0000.avi
Duration of video [s]:  2282.07 , recorded with  30.0 fps!
Overall # of frames:  68462  found with (before cropping) frame dimensions:  1440 1080
Starting to extract posture
Cropping based on the x1 = 0 x2 = 1440 y1 = 0 y2 = 1080. You can adjust the cropping coordinates in the config.yaml file.


100%|██████████| 68462/68462 [44:28<00:00, 25.65it/s]


Saving results in Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_2\4441\4441_exposure2_phase3...
Saving csv poses!


...

c:\Users\schafferlab\anaconda3\envs\DEEPLABCUT\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Starting to analyze %  Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_2\4446\4446_exposure4_phase3\4446_newport4_learn-09052025122728-0000.avi
Loading  Y:\laura_berkowitz\behavior_validation\appps1_cheeseboard\data\cohort_2\4446\4446_exposure4_phase3\4446_newport4_learn-09052025122728-0000.avi
Duration of video [s]:  1958.53 , recorded with  30.0 fps!
Overall # of frames:  58756  found with (before cropping) frame dimensions:  1440 1080
Starting to extract posture
Cropping based on the x1 = 0 x2 = 1440 y1 = 0 y2 = 1080. You can adjust the cropping coordinates in the config.yaml file.


 83%|████████▎ | 48888/58756 [16:13:03<2832:57:24, 1033.51s/it]

: 

: 